## CSV Ingestion - Load Hackerrank SQL Practice Data


### Instructions
1. Load data to each catalog volume: Use this template to query tables in Hackerrank and paste them into CSV files using notepad or other text editor.

<br>
<pre><code><b>SELECT</b> 'ID,NAME,COUNTRYCODE,DISTRICT,POPULATION' AS csv_row
<b>UNION ALL</b>
<b>SELECT</b> CONCAT(ID, ',', NAME, ',', COUNTRYCODE, ',', DISTRICT, ',', POPULATION)
<b>FROM</b> CITY;
</code></pre>
<br>
2. Excecute each cell of this notebook.
3. Check that Delta tables are successfuly created.
4. Go ahead and start working on hr_practice.

In [0]:
-- Checks for volume path and files
LIST "/Volumes/dev_world/bronze/raw_hackerrank/cities"

In [0]:
-- Checks for files content (Esto es solo consultar los datos del volumen sin crear nada en Databricks)
SELECT * FROM csv.`/Volumes/dev_world/bronze/raw_hackerrank/cities/*.csv`;

In [0]:
-- Creates CITY table with additional _metadata (Acá sí creamos la tabla delta)
CREATE OR REPLACE TABLE dev_world.bronze.city AS
SELECT 
    cast(id as INT) as id,
    cast(name as STRING) as name,
    cast(countrycode as STRING) as countrycode,
    cast(district as STRING) as district,
    cast(population as INT) as population,
    current_timestamp() AS _ingestion_time,
    _metadata.file_name AS _source_file
FROM read_files(
    '/Volumes/dev_world/bronze/raw_hackerrank/cities/',
    format => 'csv',
    header => true
);

-- Consulta los primeros 10 registros
SELECT * FROM dev_world.bronze.city LIMIT 10;

In [0]:
-- Let's create a silver Squema for Cities
CREATE SCHEMA IF NOT EXISTS dev_world.silver;

-- And a volume inside that layer
CREATE VOLUME IF NOT EXISTS dev_world.silver.city;

In [0]:
-- Creates a siver CITY table with additional _metadata (Silver no debería ingestar datos pero bueno, se entiende que esos datos evolucionaron y por eso están en esta instancia :) )
CREATE OR REPLACE TABLE dev_world.silver.city AS
SELECT 
    cast(id as INT) as id,
    cast(name as STRING) as name,
    cast(countrycode as STRING) as countrycode,
    cast(district as STRING) as district,
    cast(population as INT) as population,
    current_timestamp() AS _ingestion_time,
    _metadata.file_name AS _source_file
FROM read_files(
    '/Volumes/dev_world/silver/city/',
    format => 'csv',
    header => true
);

-- Consulta los primeros 10 registros
SELECT * FROM dev_world.silver.city LIMIT 10;

In [0]:
%python
# Lets create a dir inside bronze volume for station data using Python's standard os library on the DBFS path
import os
os.makedirs("/Volumes/dev_world/bronze/raw_hackerrank/station", exist_ok=True)

In [0]:
-- Creates STATION table with additional _metadata inside bronze schema
-- Another difference between mySQL (DECIMAL(10,0)) and Databricks SQL (DOUBLE) 
CREATE OR REPLACE TABLE dev_world.bronze.station AS
SELECT 
    cast(id as INT) as id,
    cast(city as STRING) as city,
    cast(state as STRING) as state,
    cast(lat_n as DOUBLE) as lat_n,
    cast(long_w as DOUBLE) as long_w,
    current_timestamp() AS _ingestion_time,
    _metadata.file_name AS _source_file
FROM read_files(
    '/Volumes/dev_world/bronze/raw_hackerrank/station/',
    format => 'csv',
    header => true
);

-- Consulta los primeros 10 registros
SELECT * FROM dev_world.bronze.station LIMIT 10;

In [0]:
-- Checks for volume path and files
LIST "/Volumes/dev_world/bronze/raw_hackerrank/students"

In [0]:
-- Checks for files content (Esto es solo consultar los datos del volumen sin crear nada en Databricks)
SELECT * FROM csv.`/Volumes/dev_world/bronze/raw_hackerrank/students/*.csv`;

In [0]:
-- Creates Students table with additional _metadata inside bronze schema
CREATE OR REPLACE TABLE dev_world.bronze.students AS
SELECT 
    cast(id as INT) as id,
    cast(name as STRING) as name,
    cast(marks as INT) as marks,
    current_timestamp() AS _ingestion_time,
    _metadata.file_name AS _source_file
FROM read_files(
    '/Volumes/dev_world/bronze/raw_hackerrank/students/',
    format => 'csv',
    header => true
);

-- Consulta los primeros 10 registros
SELECT * FROM dev_world.bronze.students LIMIT 10;

In [0]:
-- Creates Employees table with additional _metadata inside bronze schema
CREATE OR REPLACE TABLE dev_world.bronze.employees AS
SELECT 
    cast(employee_id as INT) as employee_id,
    cast(name as STRING) as name,
    cast(months as INT) as months,
    cast(salary as INT) as salary,
    current_timestamp() AS _ingestion_time,
    _metadata.file_name AS _source_file
FROM read_files(
    '/Volumes/dev_world/bronze/raw_hackerrank/employees/',
    format => 'csv',
    header => true
);

-- Consulta los primeros 10 registros
SELECT * FROM dev_world.bronze.employees LIMIT 10;

In [0]:
-- Checks for files content (Esto es solo consultar los datos del volumen sin crear nada en Databricks)
SELECT * FROM csv.`/Volumes/dev_world/bronze/raw_hackerrank/triangles/*.csv`;

In [0]:
-- Creates Triangles table with additional _metadata inside bronze schema
CREATE OR REPLACE TABLE dev_world.bronze.triangles AS
SELECT 
    cast(A as INT) as side_a,
    cast(B as INT) as side_b,
    cast(C as INT) as side_c,
    current_timestamp() AS _ingestion_time,
    _metadata.file_name AS _source_file
FROM read_files(
    '/Volumes/dev_world/bronze/raw_hackerrank/triangles/',
    format => 'csv',
    header => true
);

-- Consulta los primeros 10 registros
SELECT * FROM dev_world.bronze.triangles LIMIT 10;

In [0]:
-- Checks for files content (Esto es solo consultar los datos del volumen sin crear nada en Databricks)
SELECT * FROM csv.`/Volumes/dev_world/bronze/raw_hackerrank/occupations/*.csv`;

In [0]:
-- Creates Occupations table with additional _metadata inside bronze schema
CREATE OR REPLACE TABLE dev_world.bronze.occupations AS
SELECT 
    cast(Name as STRING) as name,
    cast(Occupation as STRING) as occupation,
    current_timestamp() AS _ingestion_time,
    _metadata.file_name AS _source_file
FROM read_files(
    '/Volumes/dev_world/bronze/raw_hackerrank/occupations/',
    format => 'csv',
    header => true
);

-- Consulta los primeros 10 registros
SELECT * FROM dev_world.bronze.occupations LIMIT 10;

In [0]:
-- And a volume inside silver layer
CREATE VOLUME IF NOT EXISTS dev_world.silver.occupations;

In [0]:
-- Creates Occupations table with additional _metadata inside silver schema
CREATE OR REPLACE TABLE dev_world.silver.occupations AS
SELECT 
    cast(Name as STRING) as name,
    cast(Occupation as STRING) as occupation,
    current_timestamp() AS _ingestion_time,
    _metadata.file_name AS _source_file
FROM read_files(
    '/Volumes/dev_world/silver/occupations/',
    format => 'csv',
    header => true
);

-- Consulta los primeros 10 registros
SELECT * FROM dev_world.silver.occupations ORDER BY `_source_file` DESC LIMIT 10;

In [0]:
-- Creates GDP table with additional _metadata inside bronze schema
CREATE OR REPLACE TABLE dev_world.bronze.gdp AS
SELECT 
    cast(Country as STRING) as Country,
    cast(Year as INT) as Year,
    cast(GDP_USD_Billions as DOUBLE) as GDP_USD_Billions,
    current_timestamp() AS _ingestion_time,
    _metadata.file_name AS _source_file
FROM read_files(
    '/Volumes/dev_world/bronze/raw_hackerrank/gdp/',
    format => 'csv',
    header => true
);

-- Consulta los primeros 10 registros
SELECT * FROM dev_world.bronze.gdp LIMIT 10;

In [0]:
-- Creates BST table with additional _metadata inside bronze schema
CREATE OR REPLACE TABLE dev_world.bronze.bst AS
SELECT 
    cast(N as INT) as N,
    cast(P as INT) as P,
    current_timestamp() AS _ingestion_time,
    _metadata.file_name AS _source_file
FROM read_files(
    '/Volumes/dev_world/bronze/raw_hackerrank/bst/',
    format => 'csv',
    header => true
);

-- Consulta los primeros 10 registros
SELECT * FROM dev_world.bronze.bst LIMIT 10;

In [0]:
-- Let's create a CATALOG for Amber CO
CREATE CATALOG IF NOT EXISTS amber_co MANAGED LOCATION 'abfss://unity-catalog-storage@dbstoragexjhzcdbi76pdg.dfs.core.windows.net/7405606673546676';

-- Let's create a bronze Squema for amber_co
CREATE SCHEMA IF NOT EXISTS amber_co.bronze;

-- And a volume inside that layer
CREATE VOLUME IF NOT EXISTS amber_co.bronze.hackerrank_raw;

In [0]:
-- Creates COMPANY table with additional _metadata inside bronze schema
CREATE OR REPLACE TABLE amber_co.bronze.company AS
SELECT 
    cast(company_code as STRING) as company_code,
    cast(founder as STRING) as founder,
    current_timestamp() AS _ingestion_time,
    _metadata.file_name AS _source_file
FROM read_files(
    '/Volumes/amber_co/bronze/hackerrank_raw/company_00.csv',
    format => 'csv',
    header => true
);

-- Consulta los primeros 10 registros
SELECT * FROM amber_co.bronze.company LIMIT 10;

In [0]:
-- Creates LEAD_MANAGER table with additional _metadata inside bronze schema
CREATE OR REPLACE TABLE amber_co.bronze.lead_manager AS
SELECT 
    cast(lead_manager_code as STRING) as lead_manager_code,
    cast(company_code as STRING) as company_code,
    current_timestamp() AS _ingestion_time,
    _metadata.file_name AS _source_file
FROM read_files(
    '/Volumes/amber_co/bronze/hackerrank_raw/lead_manager_00.csv',
    format => 'csv',
    header => true
);

-- Consulta los primeros 10 registros
SELECT * FROM amber_co.bronze.lead_manager LIMIT 10;

In [0]:
-- Creates SENIOR_MANAGER table with additional _metadata inside bronze schema
CREATE OR REPLACE TABLE amber_co.bronze.senior_manager AS
SELECT
    cast(senior_manager_code as STRING) as senior_manager_code, 
    cast(lead_manager_code as STRING) as lead_manager_code,
    cast(company_code as STRING) as company_code,
    current_timestamp() AS _ingestion_time,
    _metadata.file_name AS _source_file
FROM read_files(
    '/Volumes/amber_co/bronze/hackerrank_raw/senior_manager_00.csv',
    format => 'csv',
    header => true
);

-- Consulta los primeros 10 registros
SELECT * FROM amber_co.bronze.senior_manager LIMIT 10;

In [0]:
-- Creates MANAGER table with additional _metadata inside bronze schema
CREATE OR REPLACE TABLE amber_co.bronze.manager AS
SELECT
    cast(manager_code as STRING) as manager_code, 
    cast(senior_manager_code as STRING) as senior_manager_code, 
    cast(lead_manager_code as STRING) as lead_manager_code,
    cast(company_code as STRING) as company_code,
    current_timestamp() AS _ingestion_time,
    _metadata.file_name AS _source_file
FROM read_files(
    '/Volumes/amber_co/bronze/hackerrank_raw/manager_00.csv',
    format => 'csv',
    header => true
);

-- Consulta los primeros 10 registros
SELECT * FROM amber_co.bronze.manager LIMIT 10;

In [0]:
-- Creates EMPLOYEE table with additional _metadata inside bronze schema
CREATE OR REPLACE TABLE amber_co.bronze.employee AS
SELECT
    cast(employee_code as STRING) as employee_code, 
    cast(manager_code as STRING) as manager_code, 
    cast(senior_manager_code as STRING) as senior_manager_code, 
    cast(lead_manager_code as STRING) as lead_manager_code,
    cast(company_code as STRING) as company_code,
    current_timestamp() AS _ingestion_time,
    _metadata.file_name AS _source_file
FROM read_files(
    '/Volumes/amber_co/bronze/hackerrank_raw/employee_00.csv',
    format => 'csv',
    header => true
);

-- Consulta los primeros 10 registros
SELECT * FROM amber_co.bronze.employee LIMIT 10;

In [0]:
-- Creates NEW_EMPLOYEES table with additional _metadata inside bronze schema
CREATE OR REPLACE TABLE dev_world.bronze.new_employees AS
SELECT
    cast(id as INT) as ID, 
    cast(NAME as STRING) as NAME, 
    cast(SALARY as INT) as SALARY, 
    current_timestamp() AS _ingestion_time,
    _metadata.file_name AS _source_file
FROM read_files(
    '/Volumes/dev_world/bronze/raw_hackerrank/new_employees',
    format => 'csv',
    header => true
);

-- Consulta los primeros 10 registros
SELECT * FROM dev_world.bronze.new_employees LIMIT 10;